# Goal is to practice mlflow and the data

## 1. Students data

In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
path_to_data = '/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/data_n_notebook/data/Synthetic_user_data'

# read df_students.csv 
users_df = pd.read_csv(f'{path_to_data}/df_students.csv')

# read df_courses.csv
df_users = pd.read_csv(f'{path_to_data}/df_users.csv')

# show me the number of rows and columns

print(users_df.shape)
print(df_users.shape)

print(users_df.columns)
print(users_df.head())

(23236, 18)
(23236, 4)
Index(['Name', 'Age', 'Sex', 'Major', 'Year', 'GPA', 'Hobbies', 'Country',
       'State/Province', 'Unique Quality', 'Story', 'Favourite movie',
       'Favourite Book', 'Role Model', 'Learning Style', 'interests', 'ID',
       'user_id'],
      dtype='object')
                 Name  Age     Sex                    Major       Year   GPA  \
0       Thomas Ibarra   24    Male             Biochemistry     Senior  2.83   
1  Timothy Wilson PhD   23    Male  Business Administration     Senior  2.37   
2         Nancy Brown   23  Female              Archaeology  Sophomore  3.44   
3         Donna Reyes   19  Female   Mechanical Engineering     Senior  2.35   
4       Mary Gonzales   24  Female                Sociology   Freshman  2.42   

                               Hobbies Country State/Province  \
0      ['table tennis', 'photography']     USA  Massachusetts   
1      ['snowboarding', 'ice skating']  Canada        Ontario   
2            ['dancing', 'bouldering']

## 2. Knowledge base

In [3]:
df_know = pd.read_csv('/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/data_n_notebook/data/KnowlegdeBase/KnowledgeBase.csv')

df_know.columns

Index(['oasis_code', 'OaSIS Label - Final_x', ' Active Learning',
       ' Adaptability', 'Analytical Thinking', 'Attention to Detail',
       ' Creativity', 'Concern for Others', 'Collaboration', ' Independence',
       'Innovativeness', 'Leadership', 'Social Orientation',
       'Service Orientation', 'Stress Tolerance', 'Concordance number',
       'Job title type', 'Job title text', 'Main duties',
       'OaSIS Label - Final_y', 'Reading Comprehension', 'Writing  ',
       'Numeracy ', ' Digital Literacy',
       'Oral Communication: Active Listening   ',
       'Oral Communication: Oral Comprehension   ',
       'Oral Communication: Oral Expression ', 'Critical Thinking',
       'Decision Making', 'Evaluation', 'Learning and Teaching Strategies',
       'Problem Solving', 'Systems Analysis', 'Digital Production',
       'Preventative Maintenance', 'Equipment and Tool Selection',
       'Operation and Control',
       'Operation Monitoring of Machinery and Equipment',
       'Quali

# Task

Lets map the job to the user profile and interest. Then, we will make the jobs appear on a swipe card like tinder, where the student will be able to save or not the job base on his interest. 

We gonna implement the training in MLflow and compare different language model on the performance. 

We want to practice logging the models, defining configuration, logging experiments and making sure of reproductibility

1. Embedding job descriptions and student profiles
2. Computing cosine similarity
3. Logging the experiment, metadata, and performance summary using MLflow

## Data preprocessing

🎯 Objective

Is to vectorize multimodal user profiles in a way that you can use directly for search/matching, let’s build a clean, reproducible pipeline that combines:

🧠 Textual embedding (Hugging Face model)

🧮 Structured features (numerical + categorical)

📦 Into a final unified vector per user

### Semantic Embedding (Text Fields)

In [4]:
from sentence_transformers import SentenceTransformer

# Choose text-rich fields
TEXT_FIELDS = ['Story', 'interests', 'Hobbies', 'Unique Quality',
               'Learning Style', 'Favourite movie', 'Favourite Book', 'Role Model']

def build_text_input(row):
    return " ".join(str(row[col]) for col in TEXT_FIELDS if pd.notnull(row[col]))

users_df['text_input'] = users_df.apply(build_text_input, axis=1)

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")  # or another HF-compatible encoder
text_vectors = model.encode(users_df['text_input'].tolist(), show_progress_bar=True)


/Users/philippebeliveau/miniforge3/envs/orientor/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 727/727 [02:06<00:00,  5.73it/s]


In [5]:
model.get_sentence_embedding_dimension()  # returns 384 for MiniLM

384

### Structured Feature Vectorization

Handle Categorical Features

In [7]:
from sklearn.preprocessing import OneHotEncoder

CATEGORICAL_FIELDS = ['Sex', 'Major', 'Country', 'Learning Style', 'Year']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_vectors = ohe.fit_transform(users_df[CATEGORICAL_FIELDS].fillna("missing"))
joblib.dump(ohe, "/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/ohe_Siamese.pkl")

['/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/ohe_Siamese.pkl']

Normalize Numerical Features

In [8]:
from sklearn.preprocessing import StandardScaler

NUMERIC_FIELDS = ['Age', 'GPA']
scaler = StandardScaler()
numerical_vectors = scaler.fit_transform(users_df[NUMERIC_FIELDS].fillna(0))


joblib.dump(scaler, "/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/scaler_Siamese.pkl")

['/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/scaler_Siamese.pkl']

Combine Structured Features

In [9]:
structured_vectors = np.hstack([categorical_vectors, numerical_vectors])

### Final Multimodal User Vectors

In [10]:
final_user_vectors = np.hstack([text_vectors, structured_vectors])

Reduce the size to be 384

In [ ]:
from sklearn.decomposition import PCA

target_dim = 384
pca = PCA(n_components=target_dim, random_state=42)
reduced_vectors = pca.fit_transform(final_user_vectors)

reduced_vectors.shape



joblib.dump(pca, "/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/pca384_Siamese.pkl")

IsADirectoryError: [Errno 21] Is a directory: '/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/backend/app/models/'

### Insert into pinecone 



In [36]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()  # Load environment variables from .env file

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index('recommendation-index')

In [ ]:
# Rebuild user_ids cleanly
user_ids = users_df['user_id'].astype(str).tolist()

from tqdm import tqdm

def batch_upsert(index, vectors, batch_size=100):
    for i in tqdm(range(0, len(vectors), batch_size), desc="Upserting to Pinecone"):
        batch = vectors[i:i + batch_size]
        try:
            index.upsert(batch)
        except Exception as e:
            print(f"Upsert failed at batch {i}: {e}")

metadata_fields = [
    'Name', 'Age', 'Sex', 'Major', 'Year', 'GPA', 'Hobbies', 'Country',
    'State/Province', 'Unique Quality', 'Story', 'Favourite movie',
    'Favourite Book', 'Role Model', 'Learning Style', 'interests', 'user_id'
]

to_upsert = [
    (
        str(uid),
        vec.tolist(),
        {field: users_df.iloc[i][field] for field in metadata_fields if pd.notnull(users_df.iloc[i][field])}
    )
    for i, (uid, vec) in enumerate(zip(user_ids, reduced_vectors))
]


# Re-upsert
batch_upsert(index, to_upsert, batch_size=100)


Upserting to Pinecone: 100%|██████████| 233/233 [17:17<00:00,  4.45s/it]


In [47]:
print(len(user_ids), reduced_vectors.shape[0])

23236 23236


## Vectorize knowledge base

I already vectorize my data for the OaSIS dataset. Its store in pinecone at https://oasis-minilm-index-9xr744g.svc.aped-4627-b74a.pinecone.io


🧪 1. Experiment with Different Models
You can now train different types of models:

🔸 Similarity function learning: MLP on top of embeddings

🔸 Contrastive learning: Siamese/Triplet-style loss for alignment

🔸 Fine-tuned sentence transformers

🔸 Meta-learned scorers: classifier predicting relevance based on user-job vector pairs

Each model receives:

Inputs: (user_vector, job_vector)

Target: human rating, synthetic match score, or pseudo-labels

### Siamese BERT


It helps you:
Learn semantic alignment between users and jobs directly

Customize the vector space for your own domain (career guidance)

Replace generic text encoders with fine-tuned domain-aware embeddings

In your case:

One side = multimodal user description (as text)

Other side = OaSIS job description

You get a custom encoder that understands “what makes a good fit”

In [70]:
import os
from tqdm import tqdm
from pinecone import Pinecone
from sentence_transformers import InputExample
import numpy as np
import random
from pinecone import FetchResponse


def generate_siamese_pairs_from_cross_indexes(user_index_name, job_index_name, top_k=5, num_neg=5):
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

    # Connect to both indexes
    user_index = pc.Index(user_index_name)
    job_index = pc.Index(job_index_name)

    # Fetch all user vector IDs (assumes they are upserted by ID)
    user_ids = users_df["user_id"].astype(str).tolist()


    training_data = []

    for uid in tqdm(user_ids, desc="Generating pairs"):
        res = user_index.fetch(ids=[uid])
        if not hasattr(res, "vectors") or uid not in res.vectors:
            continue

        vec_data = res.vectors[uid]
        user_vec = vec_data.values
        user_text = vec_data.metadata.get("user_text", "") if vec_data.metadata else ""

        if not user_vec or not user_text:
            continue

        # 2. Query job index with user vector
        res = job_index.query(vector=user_vec, top_k=top_k + num_neg, include_metadata=True)

        # 3. POSITIVE pairs
        for match in res["matches"][:top_k]:
            job_text = match["metadata"].get("job_text", "")
            if job_text:
                training_data.append(InputExample(texts=[user_text, job_text], label=1.0))

        # 4. NEGATIVE pairs
        neg_pool = res["matches"][top_k:]
        random.shuffle(neg_pool)
        for match in neg_pool[:num_neg]:
            job_text = match["metadata"].get("job_text", "")
            if job_text:
                training_data.append(InputExample(texts=[user_text, job_text], label=0.0))

    return training_data


In [1]:
user_vec = user_index.fetch(ids=[uid])["vectors"][uid]["values"]

len(user_vec) == 384

NameError: name 'user_index' is not defined

# Mlflow

In [1]:
import mlflow

mlflow.set_experiment("JobUserSiameseMatching")

with mlflow.start_run(run_name="manual_model_upload"):
    mlflow.log_artifacts(
        local_dir="/Users/philippebeliveau/Desktop/Notebook/Orientor_project/Orientor_project/model_save/finetuned_model",
        artifact_path="model"
    )

print("✅ Model logged to MLflow. Your local model remains safe and unchanged.")


2025/04/25 09:45:00 INFO mlflow.tracking.fluent: Experiment with name 'JobUserSiameseMatching' does not exist. Creating a new experiment.


✅ Model logged to MLflow. Your local model remains safe and unchanged.


## 🔁 Push Your Fine-Tuned Model into Pinecone
Encode all users again using the fine-tuned model

Upsert to recommendation-index

Compare retrieval performance vs. original vectors (using reranking or human eval)

✅ Benefit: Validate that your training improved matching in real-world queries.



## 📡 Serve Model via API
Use FastAPI or Flask to expose:

/vectorize_user – returns 384-dim user vector

/recommend_jobs – queries Pinecone and returns metadata

Optionally add model.predict_score(user_text, job_text) endpoint